In [1]:
import dimcli
from tqdm import tqdm_notebook as tqdm
import pandas as pd
import scipy.stats

dimcli.login(instance="integration")

DimCli v0.5.9.2 - Succesfully connected to <https://integration.ds-metrics.com> (method: dsl.ini file)


In [2]:
dsl = dimcli.Dsl()

In [3]:
pubsWIthTerms = dsl.query('search publications where terms is not empty return publications[id+terms] limit 10')['publications']

In [4]:
def getReviewers(extracted_terms, field='noun_phrases', min_should_match=None, publications_limit=None):
    terms_formatted = ',\n\t\t'.join([f'"{et}"' for et in extracted_terms])
    
    query = ""
    if publications_limit is not None:
        query += f"set publications_limit={int(publications_limit)}\n"
    
    if min_should_match is not None:
        query += f"set min_should_match={int(min_should_match)}\n"
        
    query += f"""
identify reviewers for {field} [{terms_formatted}]
based on publications
where research_org_countries is not empty
    and year >= 2013
limit 200 skip 0
    """
    reviewers_result = dsl.query(query).data
    reviewers_list = sorted([{'id': _['id'], 'score': _['score']} for _ in reviewers_result['reviewer_candidates']] if 'reviewer_candidates' in reviewers_result else [], key=lambda _: _['score'], reverse=True)
    doc_list = reviewers_result['publications'] if 'publications' in reviewers_result else []
    doc_list = sorted([{'id': _['id'], 'score':_['score']} for _ in doc_list], key=lambda _: _['score'], reverse=True)
    doc_count = reviewers_result['_stats']['total_count']
    max_pub_score = max([_['score'] for _ in doc_list]) if doc_count else 1
    return query, doc_count, reviewers_list, doc_list, max_pub_score

In [5]:
results = []
i = 1
for pub in pubsWIthTerms:
    print(i)
    i+=1
    terms = pub['terms']
    
    query_relevancy, n_docs_relevancy, reviewers_relevancy, publications_relevancy, pub_max_score_relevancy = getReviewers(terms)
    query_top_n, n_docs_top_n, reviewers_top_n, publications_top_n, pub_max_score_top_n = getReviewers(terms, publications_limit=500)
    docs_count = min(n_docs_relevancy, n_docs_top_n)
    reviewers_count = min(len(reviewers_relevancy), len(publications_top_n))
    actual_publications_relevancy = publications_relevancy[:docs_count]
    actual_publications_top_n = publications_top_n[:docs_count]
    
    pearson = scipy.stats.pearsonr([pub['score'] / pub_max_score_relevancy for pub in actual_publications_relevancy], [pub['score'] / pub_max_score_top_n for pub in actual_publications_top_n]) if docs_count >= 2 else None
    spearman = scipy.stats.spearmanr([rev['score'] for rev in reviewers_relevancy], [rev['score'] for rev in reviewers_top_n])
    
    overlap_docs = len(set([_['id'] for _ in actual_publications_relevancy]).intersection(set([_['id'] for _ in actual_publications_top_n])))
    overlap_reviewers = len(set([_['id'] for _ in reviewers_relevancy]).intersection(set([_['id'] for _ in reviewers_top_n])))

    results.append({'terms': terms,
                    'query_relevancy': query_relevancy,
                    'query_top_n': query_top_n,
                    'n_docs_relevancy': n_docs_relevancy,
                    'n_docs_top_n': n_docs_top_n,
                    'reviewers_relevancy': reviewers_relevancy,
                    'reviewers_top_n': reviewers_top_n,
                    'actual_publications_relevancy': actual_publications_relevancy,
                    'actual_publications_top_n': actual_publications_top_n,
                    'pearson': pearson,
                    'spearman': spearman,
                    'overlap_docs': (overlap_docs / docs_count) * 100 if docs_count else 0,
                    'overlap_reviewers': (overlap_reviewers / reviewers_count * 100) if reviewers_count else 0})

pd.DataFrame(results)

1
2
3
4
5
6
7
8
9
10


,terms,query_relevancy,query_top_n,n_docs_relevancy,n_docs_top_n,reviewers_relevancy,reviewers_top_n,actual_publications_relevancy,actual_publications_top_n,pearson,spearman,overlap_docs,overlap_reviewers
0,"[women, memory]","\nidentify reviewers for noun_phrases [""women""...",set publications_limit=500\n\nidentify reviewe...,102904,500,"[{'id': 'ur.013410152614.91', 'score': 304.824...","[{'id': 'ur.0610125402.06', 'score': 3.4275923...","[{'id': 'pub.1112927424', 'score': 0.73849726}...","[{'id': 'pub.1105177702', 'score': 1.0415267},...","(0.9373106807398335, 3.9347547155430364e-230)","(0.9998961165534967, 0.0)",9.8,13.5
1,[memory],"\nidentify reviewers for noun_phrases [""memory...",set publications_limit=500\n\nidentify reviewe...,102974,500,"[{'id': 'ur.013410152614.91', 'score': 1796.46...","[{'id': 'ur.0610125402.06', 'score': 20.202018...","[{'id': 'pub.1112927424', 'score': 4.344223}, ...","[{'id': 'pub.1105177702', 'score': 6.135078}, ...","(0.9363363573791506, 1.617156781817759e-228)","(0.9998961165534967, 0.0)",9.8,13.5
2,"[Western Europe, Europe]","\nidentify reviewers for noun_phrases [""Wester...",set publications_limit=500\n\nidentify reviewe...,69845,500,"[{'id': 'ur.014564735611.41', 'score': 10.0118...","[{'id': 'ur.010542606715.40', 'score': 1.45013...","[{'id': 'pub.1113281594', 'score': 0.56963426}...","[{'id': 'pub.1106120862', 'score': 0.8055845},...","(0.9501738777996447, 2.9531556707336526e-254)","(0.9985679377499392, 9.227322739024185e-254)",9.2,6.5
3,"[II, metamorphosis]","\nidentify reviewers for noun_phrases [""II"",\n...",set publications_limit=500\n\nidentify reviewe...,3504,500,"[{'id': 'ur.016464742577.70', 'score': 12.9906...","[{'id': 'ur.01316133271.92', 'score': 6.799613...","[{'id': 'pub.1041776844', 'score': 1.5001948},...","[{'id': 'pub.1041776844', 'score': 1.5001948},...","(1.0, 0.0)","(0.9999246190096641, 0.0)",99.0,50.5
4,[IX],"\nidentify reviewers for noun_phrases [""IX""]\n...",set publications_limit=500\n\nidentify reviewe...,0,0,[],[],[],[],None,"(nan, nan)",0.0,0.0
5,[interference],"\nidentify reviewers for noun_phrases [""interf...",set publications_limit=500\n\nidentify reviewe...,91896,500,"[{'id': 'ur.01025057362.63', 'score': 235.5546...","[{'id': 'ur.01073651221.62', 'score': 19.39206...","[{'id': 'pub.1113788745', 'score': 3.5354168},...","[{'id': 'pub.1047326170', 'score': 4.3709426},...","(0.9819817120839263, 0.0)","(0.9998998674835521, 0.0)",5.4,11.5
6,[XVII],"\nidentify reviewers for noun_phrases [""XVII""]...",set publications_limit=500\n\nidentify reviewe...,0,0,[],[],[],[],None,"(nan, nan)",0.0,0.0
7,[possibility],"\nidentify reviewers for noun_phrases [""possib...",set publications_limit=500\n\nidentify reviewe...,245200,500,"[{'id': 'ur.012401523745.38', 'score': 47.0739...","[{'id': 'ur.07703111772.12', 'score': 13.19805...","[{'id': 'pub.1117996586', 'score': 2.1889486},...","[{'id': 'pub.1093112713', 'score': 5.0056953},...","(0.941972335118794, 3.137719417274932e-238)","(0.9995750231395132, 5.6785519421422475e-306)",3.2,8.5
8,[labor],"\nidentify reviewers for noun_phrases [""labor""...",set publications_limit=500\n\nidentify reviewe...,36909,500,"[{'id': 'ur.016116520377.99', 'score': 120.164...","[{'id': 'ur.012531502630.65', 'score': 22.1951...","[{'id': 'pub.1113236570', 'score': 4.0748186},...","[{'id': 'pub.1027080384', 'score': 6.862113}, ...","(0.9603895280566173, 1.6539873238896963e-278)","(0.9999069932806964, 0.0)",24.8,22.0
9,[absence],"\nidentify reviewers for noun_phrases [""absenc...",set publications_limit=500\n\nidentify reviewe...,230872,500,"[{'id': 'ur.0655531334.85', 'score': 52.694473...","[{'id': 'ur.013747776412.80', 'score': 5.92334...","[{'id': 'pub.1115994451', 'score': 2.1820617},...","[{'id': 'pub.1084266892', 'score': 3.525316}, ...","(0.9509838981979568, 5.525397918477595e-256)","(0.9997835953815691, 0.0)",1.8,4.5
